In [ ]:
!pip install open-clip-torch

In [ ]:
! pip install TotalSegmentator

In [ ]:
# =============================== Colab Script ===============================
# Pseudo-ROI Generation for PROH-VQA
# Couples MedSAM (mask proposal) with BiomedCLIP (vision-language scoring)
# to produce question-conditioned pseudo-ROIs without manual annotation.
#
# Setup:
#   - Install dependencies (see requirements.txt / README).
#   - Set SLAKE_PATH to your SLAKE dataset root (containing imgs/ and test.json).
#   - Set MEDSAM_CHECKPOINT to your MedSAM vit_b checkpoint (.pth).
# ----------------------------------------------------------------------------

import os, sys, math, time, json, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import cv2
from PIL import Image
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# =============================== Utils ======================================

def to_rgb(image: np.ndarray) -> np.ndarray:
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def pil_from_np_rgb(img_rgb: np.ndarray) -> Image.Image:
    return Image.fromarray(img_rgb)

def compute_iou(mask1: np.ndarray, mask2: np.ndarray) -> float:
    inter = (mask1 & mask2).sum()
    union = (mask1 | mask2).sum()
    return float(inter) / float(max(union, 1))

def bbox_from_mask(mask: np.ndarray) -> List[int]:
    ys, xs = np.where(mask)
    if xs.size == 0:
        return [0, 0, 0, 0]
    return [int(xs.min()), int(ys.min()), int(xs.max() - xs.min()), int(ys.max() - ys.min())]

def rect_intersect_ratio_with_circle_bbox(box, img_h, img_w, circle_center, circle_r):
    x, y, w, h = box
    cx, cy = circle_center
    pts = [(x + w*fx, y + h*fy) for fx in [0.1, 0.5, 0.9] for fy in [0.1, 0.5, 0.9]]
    inside = 0
    for px, py in pts:
        if (px - cx)**2 + (py - cy)**2 <= circle_r**2:
            inside += 1
    return inside / 9.0

In [ ]:
# =============================== Config =====================================

@dataclass
class GroundingConfig:
    # Grid density presets per modality and question type
    grid_density = {
        'CT':    {'default': 10, 'chest': 10, 'abdomen': 12, 'organ': 8, 'abnormality': 14},
        'X-Ray': {'default': 8,  'chest': 8,  'organ': 8,  'abnormality': 10},
        'MRI':   {'default': 8,  'organ': 8,  'abnormality': 10},
        'default': 8
    }

    # MedSAM thresholds
    sam_score_threshold: float = 0.25
    sam_stability_threshold: float = 0.75
    points_per_batch: int = 64
    multimask_output: bool = True
    max_masks_per_point: int = 3
    min_area_ratio: float = 0.002
    min_area_fallback: int = 500
    iou_threshold: float = 0.6

    # Composite score weights
    clip_weight: float = 0.55
    sam_weight: float = 0.25
    prior_weight: float = 0.15
    area_weight: float = 0.05

    # CLIP scoring options
    use_three_windows: bool = True
    use_question_rewriting: bool = True
    use_negative_evidence: bool = True
    use_source_calibration: bool = True
    neg_weight: float = 0.5                  # negative-evidence strength (eta)

    # View weights by question type (tight, padded, soft)
    window_weights = {
        'existence':  (0.50, 0.30, 0.20),
        'location':   (0.35, 0.45, 0.20),
        'comparison': (0.40, 0.40, 0.20),
        'abnormality':(0.40, 0.40, 0.20),
        'default':    (0.45, 0.35, 0.20)
    }

    # CLIP image view strategies
    use_soft_masking: bool = True
    soft_mask_alpha: float = 0.25
    crop_weight: float = 0.45
    padded_weight: float = 0.35
    masked_weight: float = 0.20
    padding_ratio: float = 0.25

    # Question rewriting
    max_rewrite_prompts: int = 8
    use_medical_terminology: bool = True
    use_modality_specific_terms: bool = True
    use_directional_terms: bool = True

    # Visualization
    show_grid_points: bool = True
    show_prompt_boxes: bool = True
    show_all_masks: bool = False
    top_k_display: int = 5

    # Fallback and retry
    enable_fallback: bool = True
    fallback_grid_size: int = 16
    max_retry_attempts: int = 2

    # Automatic mask generation (AMG)
    use_amg: bool = True
    amg_pred_iou_thresh: float = 0.65
    amg_stability_score_thresh: float = 0.75
    amg_top_k: int = 80
    amg_min_region_area: int = 200

    # Priors
    use_anatomical_priors: bool = True
    use_modality_adaptation: bool = True
    use_spatial_priors: bool = True

    # Uncertainty
    enable_uncertainty_detection: bool = True
    uncertainty_threshold: float = 0.02
    adaptive_weight_adjustment: bool = True

    # Edge penalty and safe mask control
    use_edge_penalty: bool = True
    safe_erode_kernel: int = 11              # base erosion kernel (adapted by resolution/modality)
    ct_edge_drop_threshold: float = 0.15     # CT peripheral fragment drop threshold
    enable_src_flatness_prior: bool = True   # within-source flatness prior

In [ ]:
# =============================== Parser =====================================

class MedicalQuestionParser:
    def __init__(self, config: GroundingConfig):
        self.config = config
        self.organ_terms = {
            'liver':['liver','hepatic','liver parenchyma','hepatic tissue','hepatocytes'],
            'lung':['lung','pulmonary','lung parenchyma','pulmonary fields','alveoli','pneumonic'],
            'heart':['heart','cardiac','cardiac silhouette','myocardium','pericardium'],
            'kidney':['kidney','renal','renal parenchyma','nephric','kidneys'],
            'spleen':['spleen','splenic','spleen tissue'],
            'brain':['brain','cerebral','cerebrum','neural','cranial'],
            'stomach':['stomach','gastric','gastrium'],
            'pancreas':['pancreas','pancreatic','pancreatic tissue'],
            'bladder':['bladder','vesical','urinary bladder'],
            'spine':['spine','spinal cord','vertebrae','vertebral','spinal'],
            'intestine':['intestine','bowel','colon','intestinal','gut'],
            'bone':['bone','osseous','skeletal','bony'],
            'muscle':['muscle','muscular','musculature'],
        }
        self.patterns = {
            'existence':['contain','include','have','is there','does the image','presence of','visible'],
            'location':['where','located','position','which part','location of','situated'],
            'comparison':['biggest','largest','smaller','bigger','which is','compare','larger'],
            'health':['healthy','normal','abnormal','disease','lesion','nodule','mass'],
            'count':['how many','number of','count'],
            'identification':['what is','main organ','what organ','what structure','identify'],
            'abnormality':['abnormality','abnormal','disease','lesion','tumor','cancer','pathology']
        }
        self.directional_terms = {
            'left':['left','sinister'],
            'right':['right','dexter'],
            'upper':['upper','superior','top'],
            'lower':['lower','inferior','bottom'],
            'central':['central','middle','center'],
            'peripheral':['peripheral','outer','edge']
        }
        self.pathology_terms = [
            'cancer','tumor','mass','lesion','nodule','abnormality',
            'cardiomegaly','pneumonia','atelectasis','pleural effusion',
            'consolidation','emphysema','fibrosis','inflammation','opacity','density','calcification'
        ]
        self.modality_terms = {
            'CT':['CT scan','computed tomography','cross-sectional','axial'],
            'MRI':['MRI','magnetic resonance','T1','T2','FLAIR'],
            'X-Ray':['X-ray','radiograph','chest film','plain film']
        }

    def parse(self, question: str) -> Dict[str, Any]:
        q = question.lower()
        detected_organs = [org for org, terms in self.organ_terms.items() if any(t in q for t in terms)]
        detected_directions = [d for d, terms in self.directional_terms.items() if any(t in q for t in terms)]
        qtype = 'general'
        for k, pats in self.patterns.items():
            if any(p in q for p in pats):
                qtype = k; break
        is_abn = any(t in q for t in self.pathology_terms)
        detected_pathology = [t for t in self.pathology_terms if t in q]
        if is_abn or detected_pathology:
            strategy, grid_type = 'abnormality_detection', 'abnormality'
        elif qtype == 'comparison' and len(detected_organs) > 1:
            strategy, grid_type = 'multi_organ_comparison', 'organ'
        elif qtype in ('existence', 'identification') and detected_organs:
            strategy, grid_type = 'single_organ_detection', 'organ'
        elif qtype == 'location':
            strategy, grid_type = 'spatial_localization', 'default'
        else:
            strategy, grid_type = 'general_search', 'default'
        print(f"[Parse] type={qtype} | organs={detected_organs} | directions={detected_directions}"
              f" | pathology={detected_pathology} | strategy={strategy} | grid={grid_type}")
        return {
            'original': question, 'type': qtype, 'organs': detected_organs, 'directions': detected_directions,
            'pathology': detected_pathology, 'is_abnormality': is_abn, 'strategy': strategy, 'grid_type': grid_type
        }

    def generate_prompts(self, question_info: Dict, modality_hint: Optional[str]=None) -> List[str]:
        cfg = self.config
        prompts = [question_info['original']]
        if not cfg.use_question_rewriting:
            return prompts
        for organ in question_info['organs']:
            if organ in self.organ_terms:
                prompts.extend(self.organ_terms[organ][:3])
        if cfg.use_directional_terms:
            for direction in question_info['directions']:
                if question_info['organs']:
                    for organ in question_info['organs'][:2]:
                        prompts.append(f"{direction} {organ}")
                else:
                    prompts.append(f"{direction} anatomical region")
        for pathology in question_info['pathology']:
            if cfg.use_medical_terminology:
                prompts.append(f"{pathology} in medical imaging")
                if pathology == 'cancer':
                    prompts += ['malignant lesion', 'oncological finding']
                elif pathology == 'cardiomegaly':
                    prompts += ['enlarged heart', 'cardiac enlargement']
        t = question_info['type']
        if t == 'abnormality':  prompts += ['pathological finding', 'abnormal tissue']
        elif t == 'location':   prompts += ['anatomical location', 'spatial distribution']
        elif t == 'comparison': prompts += ['comparative anatomy', 'size comparison']
        if modality_hint and self.config.use_modality_specific_terms:
            if modality_hint in self.modality_terms:
                prompts += [f"{term} anatomy" for term in self.modality_terms[modality_hint][:2]]
        if not question_info['organs']:
            if modality_hint in ['X-Ray', 'CT']:
                prompts += ['chest anatomy', 'thoracic structures', 'abdominal organs']
            else:
                prompts += ['human anatomy', 'organ structures']
        seen = set(); out = []
        for p in prompts:
            pl = p.lower()
            if pl not in seen:
                out.append(p); seen.add(pl)
        out = out[:self.config.max_rewrite_prompts]
        print(f"[Parse] positive prompts ({len(out)}): {out}")
        return out

    def generate_negative_prompts(self, qinfo: Dict, modality_hint: Optional[str]=None) -> List[str]:
        if not self.config.use_negative_evidence: return []
        if qinfo['type'] not in ['existence', 'health']: return []
        neg = []
        for organ in qinfo['organs']:
            neg += [f"medical image without {organ}", f"non-{organ} anatomy", f"other organs not {organ}"]
        neg += ["background medical tissue", "anatomical background", "non-target structure"]
        print(f"[Parse] negative prompts: {neg[:4]}")
        return neg[:4]

In [ ]:
# =============================== Preprocessor ===============================

class ImagePreprocessor:
    def __init__(self):
        pass

    def detect_modality(self, bgr: np.ndarray) -> Tuple[str, Dict[str, Any]]:
        h, w = bgr.shape[:2]
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        csz = max(8, min(h, w) // 10)
        corners = [gray[0:csz, 0:csz], gray[0:csz, -csz:], gray[-csz:, 0:csz], gray[-csz:, -csz:]]
        corner_mean = np.mean([c.mean() for c in corners])
        edge = cv2.Canny(gray, 20, 80)
        edges_ratio = float(edge.sum() > 0) and (edge.sum() / 255.0) / (h * w)
        dark_ratio = (gray < 8).sum() / (h * w)
        is_ct = (corner_mean < 12 and dark_ratio > 0.05) or (edges_ratio < 0.03 and corner_mean < 15)
        if is_ct:
            return 'CT', {'corner_mean': corner_mean}
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
        high_tail = hist[240:].sum() / (h * w)
        std_int = gray.std()
        is_xray = (high_tail > 0.005 and std_int > 40) or (high_tail > 0.002 and gray.mean() < 110)
        if is_xray:
            return 'X-Ray', {'high_tail': float(high_tail), 'std': float(std_int)}
        return 'MRI', {'std': float(std_int), 'mean': float(gray.mean())}

    def build_body_mask(self, bgr: np.ndarray, modality: str) -> Tuple[np.ndarray, Optional[Tuple[int,int,int]]]:
        h, w = bgr.shape[:2]
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        if modality == 'CT':
            thr = 8
            body = (gray > thr).astype(np.uint8) * 255
            body = cv2.morphologyEx(body, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
            cnts, _ = cv2.findContours(body, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts:
                return (gray > 0), None
            c = max(cnts, key=cv2.contourArea)
            mask = np.zeros((h, w), np.uint8); cv2.drawContours(mask, [c], -1, 255, -1)
            (cx, cy), r = cv2.minEnclosingCircle(c)
            return (mask > 0), (int(cx), int(cy), int(r))
        else:
            _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            if th.mean() > 127:
                th = 255 - th
            th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
            cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts:
                return (gray > 0), None
            c = max(cnts, key=cv2.contourArea)
            mask = np.zeros((h, w), np.uint8); cv2.drawContours(mask, [c], -1, 255, -1)
            return (mask > 0), None

    def enhance_for_modality(self, bgr: np.ndarray, modality: str) -> np.ndarray:
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        if modality in ['CT', 'X-Ray']:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            eq = clahe.apply(gray)
            return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)
        else:
            g = gray.astype(np.float32)
            g = (g - g.mean()) / (g.std() + 1e-6) if g.std() >= 1e-6 else (g - g.min())
            g = (g - g.min()) / (g.max() - g.min() + 1e-6)
            g = (g * 255.0).clip(0, 255).astype(np.uint8)
            return cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)

In [ ]:
# =============================== MedSAM Processor ===========================

class MedSAMProcessor:
    def __init__(self, checkpoint_path: str, device: str='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        print(f"[MedSAM] loading from: {checkpoint_path}")
        from segment_anything import sam_model_registry, SamPredictor
        self.sam_model = sam_model_registry['vit_b'](checkpoint=checkpoint_path).to(self.device).eval()
        self.predictor = SamPredictor(self.sam_model)
        print("[MedSAM] loaded")
        try:
            from segment_anything import SamAutomaticMaskGenerator
            self.SamAutomaticMaskGenerator = SamAutomaticMaskGenerator
        except Exception:
            self.SamAutomaticMaskGenerator = None
        self.grid_points_viz = None
        self.prompt_boxes_viz = None
        self.body_mask_info = None

    def set_image(self, rgb: np.ndarray):
        self.predictor.set_image(rgb)

    # ---------- grid ----------
    def generate_grid_masks(self, rgb: np.ndarray, body_mask: np.ndarray, modality: str,
                            qinfo: Dict, cfg: GroundingConfig) -> List[Dict]:
        h, w = rgb.shape[:2]
        coverage = body_mask.sum() / (h * w)
        print(f"[Grid] body mask coverage: {coverage:.3f}")
        self.body_mask_info = {'coverage': coverage}
        if coverage < 0.3 and cfg.enable_fallback:
            print("[Grid] low coverage, falling back to full image")
            body_mask = np.ones((h, w), dtype=bool)

        grid_type = qinfo.get('grid_type', 'default')
        base_grid = cfg.grid_density.get(modality, {}).get(grid_type,
                    cfg.grid_density.get(modality, {}).get('default', cfg.grid_density['default']))
        if qinfo.get('is_abnormality', False):
            grid_size = min(base_grid + 2, 16)
        elif len(qinfo.get('organs', [])) > 1:
            grid_size = min(base_grid + 1, 14)
        else:
            grid_size = base_grid
        print(f"[Grid] size {grid_size}x{grid_size} (modality={modality}, type={grid_type})")

        points = []
        for i in range(grid_size):
            for j in range(grid_size):
                x = int((j + 0.5) * w / grid_size)
                y = int((i + 0.5) * h / grid_size)
                if body_mask[y, x]:
                    points.append([x, y])

        if qinfo.get('organs'):
            anc = self._generate_organ_anchor_points(h, w, body_mask, qinfo['organs'], modality)
            points.extend(anc)
            print(f"[Grid] added {len(anc)} organ-specific anchors")

        if len(points) < 10:
            print(f"[Grid] too few points ({len(points)}), applying fallback")
            kernel = np.ones((11, 11), np.uint8)
            expanded = cv2.dilate(body_mask.astype(np.uint8), kernel, 1)
            gs = max(grid_size, cfg.fallback_grid_size)
            for i in range(gs):
                for j in range(gs):
                    x = int((j + 0.5) * w / gs)
                    y = int((i + 0.5) * h / gs)
                    if expanded[y, x] and [x, y] not in points:
                        points.append([x, y])

        self.grid_points_viz = np.array(points) if points else None
        print(f"[Grid] generated {len(points)} points")

        masks = []
        if not points:
            return masks
        min_area = max(cfg.min_area_fallback, int(h * w * cfg.min_area_ratio))
        print(f"[Grid] dynamic area threshold: {min_area}")
        pts = np.array(points, dtype=np.float32)
        labels = np.ones((len(points),), dtype=np.int32)
        bs = cfg.points_per_batch; total = 0
        for s in range(0, len(points), bs):
            pe = min(s + bs, len(points))
            batch_points = pts[s:pe]; batch_labels = labels[s:pe]
            for p, lb in zip(batch_points, batch_labels):
                try:
                    m, scores, _ = self.predictor.predict(point_coords=p[None, :],
                                   point_labels=np.array([lb]), multimask_output=cfg.multimask_output)
                    for mk, sc in zip(m, scores):
                        area = int(mk.sum())
                        if sc > cfg.sam_score_threshold and area > min_area:
                            masks.append({'segmentation': mk.astype(bool), 'score': float(sc), 'area': area,
                                          'source': 'grid', 'prompt_point': [int(p[0]), int(p[1])]})
                            total += 1
                except Exception:
                    continue
        print(f"[Grid] produced {total} masks")
        return masks

    def _generate_organ_anchor_points(self, h:int, w:int, body_mask:np.ndarray,
                                       organs:List[str], modality:str) -> List[List[int]]:
        anchor_points = []
        organ_regions = {
            'lung':  [(0.2, 0.2, 0.45, 0.8), (0.55, 0.2, 0.8, 0.8)],
            'heart': [(0.4, 0.35, 0.6, 0.7)],
            'liver': [(0.5, 0.4, 0.9, 0.8)],
            'spleen':[(0.1, 0.4, 0.5, 0.75)],
            'kidney':[(0.2, 0.4, 0.8, 0.8)]
        }
        for organ in organs:
            if organ in organ_regions:
                for (x0, y0, x1, y1) in organ_regions[organ]:
                    for dx, dy in [(0.3, 0.3), (0.5, 0.5), (0.7, 0.7)]:
                        x = int((x0 + (x1 - x0) * dx) * w)
                        y = int((y0 + (y1 - y0) * dy) * h)
                        x = max(0, min(x, w - 1)); y = max(0, min(y, h - 1))
                        if body_mask[y, x]:
                            anchor_points.append([x, y])
        return anchor_points

    # ---------- boxes ----------
    def generate_box_masks(self, rgb: np.ndarray, body_mask: np.ndarray, modality: str,
                           qinfo: Dict, cfg: GroundingConfig) -> List[Dict]:
        h, w = rgb.shape[:2]
        ys, xs = np.where(body_mask)
        if len(xs) == 0:
            return self._generate_fallback_boxes(rgb, modality, cfg)
        bx0, bx1, by0, by1 = xs.min(), xs.max(), ys.min(), ys.max()
        cx, cy = (bx0 + bx1) // 2, (by0 + by1) // 2
        bw, bh = bx1 - bx0, by1 - by0
        print(f"[Box] body ROI: ({bx0},{by0})-({bx1},{by1})")

        boxes = []
        det = qinfo.get('organs', [])
        dirs = qinfo.get('directions', [])
        if 'lung' in det:   boxes += self._generate_lung_boxes(bx0, by0, bw, bh, dirs)
        if 'heart' in det:  boxes.append(self._generate_heart_box(cx, cy, bw, bh))
        if 'liver' in det:  boxes.append(self._generate_liver_box(bx0, by0, bw, bh))
        if 'spleen' in det: boxes.append(self._generate_spleen_box(bx0, by0, bw, bh))
        for scale in [0.3, 0.5, 0.7, 0.9]:
            ww, hh = int(bw * scale), int(bh * scale)
            x = cx - ww // 2; y = cy - hh // 2
            x = max(bx0, min(x, bx1 - ww)); y = max(by0, min(y, by1 - hh))
            boxes.append([x, y, x + ww, y + hh])
        if qinfo.get('is_abnormality', False):
            boxes += self._generate_dense_boxes(bx0, by0, bw, bh)

        self.prompt_boxes_viz = boxes.copy()
        print(f"[Box] generated {len(boxes)} boxes")

        masks = []
        for i, box in enumerate(boxes):
            try:
                m, scores, _ = self.predictor.predict(box=np.array(box, dtype=np.int32),
                                                      multimask_output=cfg.multimask_output)
                min_area = max(cfg.min_area_fallback, int(h * w * cfg.min_area_ratio))
                for mk, sc in zip(m, scores):
                    area = int(mk.sum())
                    if sc > cfg.sam_score_threshold and area > min_area:
                        masks.append({'segmentation': mk.astype(bool), 'score': float(sc), 'area': area,
                                      'source': 'box', 'prompt_box': box, 'box_index': i})
            except Exception as e:
                print(f"[Box] prediction failed at box {i}: {e}")
                continue
        print(f"[Box] produced {len(masks)} masks")
        return masks

    def _generate_lung_boxes(self, x_min, y_min, width, height, directions):
        boxes = []; ys = y_min + int(height * 0.1); ye = y_min + int(height * 0.8)
        if not directions or 'left' in directions or 'bilateral' in directions:
            boxes.append([x_min + int(width * 0.05), ys, x_min + int(width * 0.45), ye])
        if not directions or 'right' in directions or 'bilateral' in directions:
            boxes.append([x_min + int(width * 0.55), ys, x_min + int(width * 0.95), ye])
        return boxes

    def _generate_heart_box(self, cx, cy, w, h):
        return [cx - int(w * 0.15), cy - int(h * 0.2), cx + int(w * 0.15), cy + int(h * 0.2)]

    def _generate_liver_box(self, x_min, y_min, w, h):
        return [x_min + int(w * 0.5), y_min + int(h * 0.4), x_min + int(w * 0.95), y_min + int(h * 0.8)]

    def _generate_spleen_box(self, x_min, y_min, w, h):
        return [x_min + int(w * 0.05), y_min + int(h * 0.4), x_min + int(w * 0.5), y_min + int(h * 0.75)]

    def _generate_dense_boxes(self, x_min, y_min, w, h):
        boxes = []; overlap = 0.2
        for i in range(3):
            for j in range(3):
                gw = w // 3; gh = h // 3; gx = x_min + j * gw; gy = y_min + i * gh
                aw = int(gw * (1 + overlap)); ah = int(gh * (1 + overlap))
                boxes.append([max(x_min, gx - int(gw * overlap / 2)), max(y_min, gy - int(gh * overlap / 2)),
                              min(x_min + w, gx + aw), min(y_min + h, gy + ah)])
        return boxes

    def _generate_fallback_boxes(self, rgb: np.ndarray, modality: str, cfg: GroundingConfig) -> List[Dict]:
        print("[Box] using fallback boxes")
        h, w = rgb.shape[:2]; boxes = []
        for scale in [0.4, 0.6, 0.8]:
            bw, bh = int(w * scale), int(h * scale); cx, cy = w // 2, h // 2
            boxes.append([cx - bw // 2, cy - bh // 2, cx + bw // 2, cy + bh // 2])
        self.prompt_boxes_viz = boxes.copy()
        return []

    # ---------- AMG ----------
    def generate_amg_masks(self, rgb: np.ndarray, cfg: GroundingConfig) -> List[Dict]:
        if self.SamAutomaticMaskGenerator is None or not cfg.use_amg:
            return []
        print("[AMG] generating masks")
        try:
            gen = self.SamAutomaticMaskGenerator(model=self.sam_model, points_per_side=16,
                pred_iou_thresh=cfg.amg_pred_iou_thresh, stability_score_thresh=cfg.amg_stability_score_thresh,
                min_mask_region_area=cfg.amg_min_region_area)
            results = gen.generate(rgb)
            print(f"[AMG] raw masks: {len(results)}")
            results = sorted(results, key=lambda x: x.get('area', 0), reverse=True)[:cfg.amg_top_k]
            h, w = rgb.shape[:2]; min_area = max(cfg.min_area_fallback, int(h * w * cfg.min_area_ratio))
            masks = []
            for i, r in enumerate(results):
                seg = r['segmentation'].astype(bool); area = int(seg.sum())
                score = float(r.get('predicted_iou', 0.7))
                if area > min_area:
                    masks.append({'segmentation': seg, 'score': score, 'area': area, 'source': 'amg', 'amg_index': i})
            print(f"[AMG] kept: {len(masks)}")
            return masks
        except Exception as e:
            print(f"[AMG] failed: {e}")
            return []

    # ---------- dedup ----------
    def dedup(self, masks: List[Dict], iou_thr: float, cfg: GroundingConfig) -> List[Dict]:
        if len(masks) <= 1:
            return masks
        print(f"[Dedup] deduplicating {len(masks)} masks")
        by_source = {}
        for m in masks:
            by_source.setdefault(m['source'], []).append(m)
        deduped = {}
        for src, arr in by_source.items():
            if len(arr) > 1:
                arr = sorted(arr, key=lambda x: (x['score'], x['area']), reverse=True)
                keep = []
                for m in arr:
                    if all(compute_iou(m['segmentation'], k['segmentation']) <= iou_thr for k in keep):
                        keep.append(m)
                deduped[src] = keep
            else:
                deduped[src] = arr
        allm = []
        for v in deduped.values():
            allm.extend(v)
        if len(allm) > 1:
            allm = sorted(allm, key=lambda x: (x['score'], x['area']), reverse=True)
            final = []
            for m in allm:
                if all(compute_iou(m['segmentation'], k['segmentation']) <= iou_thr for k in final):
                    final.append(m)
            allm = final
        c = {}
        for m in allm:
            c[m['source']] = c.get(m['source'], 0) + 1
        print(f"[Dedup] after dedup: {len(allm)} | sources: {c}")
        return allm

In [ ]:
# =============================== BiomedCLIP Processor =======================

class BiomedCLIPProcessor:
    def __init__(self, device: str='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        print("[CLIP] loading BiomedCLIP")
        self.use_open_clip = True
        try:
            import open_clip
            model_name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
            self.model, _, self.preprocess = open_clip.create_model_and_transforms(model_name)
            self.tok = open_clip.get_tokenizer(model_name)
            self.model.to(self.device).eval()
            self.model_name = model_name
            print("[CLIP] BiomedCLIP (open_clip) loaded")
        except Exception as e:
            print(f"[CLIP] open_clip unavailable ({e}); falling back to CLIP ViT-B/32")
            from transformers import CLIPModel, CLIPProcessor
            self.use_open_clip = False
            self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device).eval()
            self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
            self.model_name = "openai/clip-vit-base-patch32"
        self.text_cache = {}

    def _encode_img(self, pil_img: Image.Image):
        if self.use_open_clip:
            tens = self.preprocess(pil_img).unsqueeze(0).to(self.device)
            with torch.no_grad():
                feats = self.model.encode_image(tens); feats = F.normalize(feats, dim=-1)
            return feats
        else:
            inputs = self.processor(images=pil_img, return_tensors="pt").to(self.device)
            with torch.no_grad():
                feats = self.model.get_image_features(**inputs); feats = F.normalize(feats, dim=-1)
            return feats

    def _encode_texts(self, prompts: List[str]):
        key = (self.model_name, tuple(sorted(prompts)))
        if key in self.text_cache:
            return self.text_cache[key]
        if self.use_open_clip:
            toks = self.tok(prompts).to(self.device)
            with torch.no_grad():
                feats = self.model.encode_text(toks); feats = F.normalize(feats, dim=-1)
        else:
            inputs = self.processor(text=prompts, return_tensors="pt", padding=True).to(self.device)
            with torch.no_grad():
                feats = self.model.get_text_features(**inputs); feats = F.normalize(feats, dim=-1)
        self.text_cache[key] = feats
        return feats

    @staticmethod
    def percentile_scale_np(x, eps=1e-6):
        # Within-source percentile calibration (rank-based, SciPy-free).
        x = np.asarray(x, dtype=np.float32); n = len(x)
        if n <= 1:
            return np.full(n, 0.5, dtype=np.float32)
        if float(x.min()) == float(x.max()):
            return np.full(n, 0.5, dtype=np.float32)
        uniq, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
        cum = np.cumsum(counts)
        ranks_hi = cum[inv]
        ranks_lo = ranks_hi - counts[inv] + 1
        ranks_avg = (ranks_lo + ranks_hi - 1) / 2.0
        scaled = (ranks_avg - 1) / (n - 1 + eps)
        return np.clip(scaled, eps, 1.0 - eps)

    def apply_source_calibration(self, masks: List[Dict], cfg: GroundingConfig) -> List[Dict]:
        if not cfg.use_source_calibration or len(masks) == 0:
            return masks
        print("[CLIP] applying within-source percentile calibration")
        by = {}
        for m in masks:
            by.setdefault(m['source'], []).append(m)
        out = []
        for src, arr in by.items():
            vals = [m['score'] for m in arr]
            scaled = self.percentile_scale_np(vals)
            for m, s in zip(arr, scaled):
                m['sam_score_original'] = m['score']
                m['score'] = float(np.clip(s, 0.0, 1.0))
            out += arr
            print(f"[CLIP] {src}: min={min(vals):.3f} max={max(vals):.3f} -> scaled to [0,1]")
        return out

    def _three_windows(self, image_rgb: np.ndarray, mask: Dict, cfg: GroundingConfig) -> Dict[str, Image.Image]:
        seg = mask['segmentation']; ys, xs = np.where(seg); h, w = image_rgb.shape[:2]
        if xs.size == 0:
            csz = min(h, w) // 2; cy, cx = h // 2, w // 2
            y0 = max(0, cy - csz // 2); y1 = min(h, cy + csz // 2)
            x0 = max(0, cx - csz // 2); x1 = min(w, cx + csz // 2)
            crop = image_rgb[y0:y1, x0:x1]; pil = pil_from_np_rgb(crop)
            return {'tight': pil, 'padded': pil, 'soft': pil}
        x0, x1 = xs.min(), xs.max(); y0, y1 = ys.min(), ys.max()
        tight = pil_from_np_rgb(image_rgb[y0:y1+1, x0:x1+1])
        pad_w = int((x1 - x0) * cfg.padding_ratio); pad_h = int((y1 - y0) * cfg.padding_ratio)
        px0 = max(0, x0 - pad_w); px1 = min(w, x1 + pad_w)
        py0 = max(0, y0 - pad_h); py1 = min(h, y1 + pad_h)
        padded = pil_from_np_rgb(image_rgb[py0:py1, px0:px1])
        if cfg.use_soft_masking:
            m3 = np.repeat(seg[..., None], 3, axis=2).astype(np.float32)
            soft = image_rgb.astype(np.float32) * m3 + image_rgb.astype(np.float32) * (1 - m3) * cfg.soft_mask_alpha
            soft = soft.clip(0, 255).astype(np.uint8); soft = pil_from_np_rgb(soft)
        else:
            soft = tight
        return {'tight': tight, 'padded': padded, 'soft': soft}

    def compute_similarity(self, image_rgb: np.ndarray, mask: Dict, prompts: List[str],
                           negative_prompts: List[str], qinfo: Dict, cfg: GroundingConfig) -> Dict[str, float]:
        wins = self._three_windows(image_rgb, mask, cfg)
        t = qinfo.get('type', 'default'); tw, pw, sw = cfg.window_weights.get(t, cfg.window_weights['default'])
        pos_txt = self._encode_texts(prompts) if len(prompts) > 0 else None
        neg_txt = self._encode_texts(negative_prompts) if (negative_prompts and cfg.use_negative_evidence) else None

        def sim_one(pil_img):
            img = self._encode_img(pil_img)
            if pos_txt is not None:
                with torch.no_grad():
                    s = (img @ pos_txt.T).squeeze(0)
                pos = float(s.max().item()) if s.numel() > 0 else 0.0
            else:
                pos = 0.0
            if neg_txt is not None:
                with torch.no_grad():
                    s = (img @ neg_txt.T).squeeze(0)
                neg = float(s.max().item()) if s.numel() > 0 else 0.0
            else:
                neg = 0.0
            return pos, neg

        tight_pos, tight_neg = sim_one(wins['tight'])
        padded_pos, padded_neg = sim_one(wins['padded'])
        soft_pos, soft_neg = sim_one(wins['soft'])

        final_pos = tw * tight_pos + pw * padded_pos + sw * soft_pos
        final_neg = tw * tight_neg + pw * padded_neg + sw * soft_neg
        clip_final = final_pos - cfg.neg_weight * final_neg

        pos_scores = [tight_pos, padded_pos, soft_pos]
        uncertainty = float(np.var(pos_scores)) if len(pos_scores) > 1 else 0.0

        return {
            'tight_pos': tight_pos, 'padded_pos': padded_pos, 'soft_pos': soft_pos,
            'final_positive': final_pos, 'final_negative': final_neg,
            'final': clip_final, 'uncertainty': uncertainty
        }

In [ ]:
# =============================== Main System ================================

class MedicalVQAGrounding:
    def __init__(self, medsam_checkpoint: str, config: Optional[GroundingConfig]=None):
        print("=" * 70)
        print("PROH-VQA Pseudo-ROI Generation")
        print("=" * 70)
        self.cfg = config or GroundingConfig()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"[Init] device: {self.device}")
        self.parser = MedicalQuestionParser(self.cfg)
        self.preproc = ImagePreprocessor()
        self.medsam = MedSAMProcessor(medsam_checkpoint, str(self.device))
        self.clip = BiomedCLIPProcessor(str(self.device))
        print("[Init] system ready")

    @staticmethod
    def force_ct_if_circle(modality: str, ct_circle: Optional[Tuple[int,int,int]]) -> str:
        # A large circular field of view indicates a CT scan; force the label accordingly.
        if ct_circle is not None:
            return 'CT'
        return modality

    @staticmethod
    def build_safe_mask(body_mask: np.ndarray, modality: str, h: int, w: int, cfg: GroundingConfig):
        # Adaptive erosion kernel scaled by resolution and modality.
        base = int(round(0.02 * min(h, w))) | 1  # odd kernel ~2% of image size
        if modality == 'CT':
            k = max(base, cfg.safe_erode_kernel)
        elif modality == 'X-Ray':
            k = max(base - 2, 7)
        else:  # MRI
            k = max(base - 4, 5)
        safe_mask = cv2.erode(body_mask.astype(np.uint8), np.ones((k, k), np.uint8), 1).astype(bool)
        if safe_mask.mean() < 0.2:  # reduce kernel if coverage too small
            k = max(5, int(0.7 * k) | 1)
            safe_mask = cv2.erode(body_mask.astype(np.uint8), np.ones((k, k), np.uint8), 1).astype(bool)
        dist = cv2.distanceTransform(safe_mask.astype(np.uint8), cv2.DIST_L2, 3)
        return safe_mask, dist

    @staticmethod
    def src_flatness_weight(source_masks: List[Dict]) -> float:
        # A small std of within-source raw SAM scores indicates a flat distribution; apply a light penalty.
        if len(source_masks) < 3:
            return 1.0
        s = np.array([m.get('sam_score_original', m['score']) for m in source_masks], dtype=np.float32)
        std = float(np.std(s))
        if std >= 0.05:
            return 1.0
        return 0.85 + 0.15 * (std / 0.05)  # std: 0 -> 0.85, 0.05 -> 1.0

    def _get_adaptive_weights(self, qinfo: Dict, sims: Dict, cfg: GroundingConfig) -> Tuple[float,float,float,float]:
        qtype = qinfo.get('type', 'default'); unc = sims.get('uncertainty', 0.0)
        clip_w, sam_w, prior_w, area_w = cfg.clip_weight, cfg.sam_weight, cfg.prior_weight, cfg.area_weight
        if qtype == 'comparison':
            clip_w, sam_w, prior_w, area_w = 0.45, 0.25, 0.10, 0.20
        elif qtype == 'existence':
            clip_w, sam_w, prior_w = 0.60, 0.25, 0.15
        elif qtype == 'location':
            clip_w, sam_w, prior_w = 0.50, 0.25, 0.25
        elif qtype == 'abnormality':
            clip_w, sam_w, prior_w = 0.55, 0.30, 0.15
        if cfg.adaptive_weight_adjustment and unc > 0.05:
            print(f"[Score] adjusting weights (uncertainty={unc:.3f})")
            clip_w *= 0.8; prior_w *= 1.4
            tot = clip_w + sam_w + prior_w
            clip_w /= tot; sam_w /= tot; prior_w /= tot
        return clip_w, sam_w, prior_w, area_w

    def _position_prior(self, seg: np.ndarray, qinfo: Dict, img_shape: Tuple[int,int], modality: str) -> float:
        h, w = img_shape; ys, xs = np.where(seg)
        if xs.size == 0:
            return 0.0
        cx, cy = xs.mean() / w, ys.mean() / h
        organ_priors = {
            'lung':  {'CT':{'x':(0.15,0.85),'y':(0.15,0.75)}, 'X-Ray':{'x':(0.20,0.80),'y':(0.20,0.70)}, 'MRI':{'x':(0.15,0.85),'y':(0.15,0.75)}},
            'heart': {'CT':{'x':(0.35,0.65),'y':(0.30,0.70)}, 'X-Ray':{'x':(0.40,0.60),'y':(0.35,0.65)}, 'MRI':{'x':(0.35,0.65),'y':(0.30,0.70)}},
            'liver': {'CT':{'x':(0.50,0.90),'y':(0.40,0.80)}, 'MRI':{'x':(0.50,0.90),'y':(0.40,0.80)}},
            'spleen':{'CT':{'x':(0.10,0.50),'y':(0.40,0.75)}, 'MRI':{'x':(0.10,0.50),'y':(0.40,0.75)}}
        }
        base = 0.5
        if qinfo.get('organs'):
            for organ in qinfo['organs']:
                if organ in organ_priors and modality in organ_priors[organ]:
                    px, py = organ_priors[organ][modality]['x'], organ_priors[organ][modality]['y']
                    sx = 1.0 if px[0] <= cx <= px[1] else 0.3
                    sy = 1.0 if py[0] <= cy <= py[1] else 0.3
                    base = max(base, (sx + sy) / 2.0)
        dirs = qinfo.get('directions', [])
        if dirs:
            if 'left' in dirs and cx < 0.5: base = max(base, 0.8)
            if 'right' in dirs and cx > 0.5: base = max(base, 0.8)
            if 'upper' in dirs and cy < 0.5: base = max(base, 0.8)
            if 'lower' in dirs and cy > 0.5: base = max(base, 0.8)
            if 'central' in dirs and 0.3 < cx < 0.7 and 0.3 < cy < 0.7: base = max(base, 0.8)
        return base

    def process_single_sample(self, image_path: str, question: str, answer: str,
                              sample_id: int=0, content_type: Optional[str]=None) -> Optional[Dict]:
        print(f"\n{'=' * 70}\n[Sample {sample_id}]\n{'=' * 70}")
        bgr = cv2.imread(image_path)
        if bgr is None:
            print(f"[Sample {sample_id}] failed to load: {image_path}")
            return None
        h, w = bgr.shape[:2]
        print(f"[Sample {sample_id}] image: {Path(image_path).name} | shape: {bgr.shape}")
        print(f"[Sample {sample_id}] question: {question}")
        print(f"[Sample {sample_id}] answer: {answer}")
        if content_type:
            print(f"[Sample {sample_id}] content type: {content_type}")

        qinfo = self.parser.parse(question)

        modality, dbg = self.preproc.detect_modality(bgr)
        body_mask, ct_circle = self.preproc.build_body_mask(bgr, modality)
        modality = self.force_ct_if_circle(modality, ct_circle)
        print(f"[Sample {sample_id}] modality: {modality} | debug: {dbg} | CT circle: {ct_circle}")

        rgb_for_sam = self.preproc.enhance_for_modality(bgr, modality)
        self.medsam.set_image(rgb_for_sam)

        prompts = self.parser.generate_prompts(qinfo, modality_hint=modality)
        negative_prompts = self.parser.generate_negative_prompts(qinfo, modality_hint=modality)

        safe_mask, dist = self.build_safe_mask(body_mask, modality, h, w, self.cfg)

        all_masks = []; retry = 0
        maxr = self.cfg.max_retry_attempts if self.cfg.enable_fallback else 1
        while retry < maxr:
            print(f"[Sample {sample_id}] mask generation attempt {retry + 1}/{maxr}")
            grid_masks = self.medsam.generate_grid_masks(rgb_for_sam, safe_mask, modality, qinfo, self.cfg)
            all_masks += grid_masks
            box_masks = self.medsam.generate_box_masks(rgb_for_sam, safe_mask, modality, qinfo, self.cfg)
            all_masks += box_masks
            if self.cfg.use_amg:
                amg_masks = self.medsam.generate_amg_masks(rgb_for_sam, self.cfg)
                all_masks += amg_masks
            if len(all_masks) >= 10:
                break
            elif retry < maxr - 1:
                print(f"[Sample {sample_id}] only {len(all_masks)} masks, relaxing thresholds")
                self.cfg.sam_score_threshold = max(0.15, self.cfg.sam_score_threshold - 0.1)
                self.cfg.min_area_ratio = max(0.0005, self.cfg.min_area_ratio * 0.5)
                all_masks = []
            retry += 1

        all_masks = self.medsam.dedup(all_masks, self.cfg.iou_threshold, self.cfg)
        if not all_masks:
            print(f"[Sample {sample_id}] no masks after dedup")
            return None

        all_masks = self.clip.apply_source_calibration(all_masks, self.cfg)

        src_weight_map = {}
        if self.cfg.enable_src_flatness_prior:
            by = {}
            for m in all_masks:
                by.setdefault(m['source'], []).append(m)
            for src, arr in by.items():
                src_weight_map[src] = self.src_flatness_weight(arr)
            print(f"[Score] source flatness weights: {src_weight_map}")

        print("[Score] computing CLIP similarities and composite scores")
        scored = []
        for i, m in enumerate(all_masks[:200]):
            try:
                sims = self.clip.compute_similarity(rgb_for_sam, m, prompts, negative_prompts, qinfo, self.cfg)
                prior = self._position_prior(m['segmentation'], qinfo, (h, w), modality)
                area = m['area']; area_norm = math.sqrt(area) / math.sqrt(h * h + w * w)
                clip_w, sam_w, prior_w, area_w = self._get_adaptive_weights(qinfo, sims, self.cfg)

                if self.cfg.enable_src_flatness_prior:
                    sam_w *= src_weight_map.get(m['source'], 1.0)

                # Drop small peripheral fragments (CT only).
                if modality == 'CT':
                    ys, xs = np.where(m['segmentation'])
                    if xs.size > 0:
                        d_mean = float(dist[ys, xs].mean()) / float(max(dist.max(), 1))
                        area_ratio = area / (h * w + 1e-6)
                        if d_mean < self.cfg.ct_edge_drop_threshold and area_ratio < 0.01:
                            continue

                sam_score = float(np.clip(m['score'], 0.0, 1.0))

                # Edge penalty applied only to the SAM term.
                sam_term = sam_w * sam_score
                if self.cfg.use_edge_penalty:
                    ys, xs = np.where(m['segmentation'])
                    if xs.size > 0:
                        d_mean = float(dist[ys, xs].mean()) / float(max(dist.max(), 1))
                        area_ratio = area / (h * w + 1e-6)
                        alpha = float(np.clip(area_ratio / 0.02, 0.0, 1.0))
                        pen_lo = 0.20 + 0.80 * d_mean
                        pen_hi = 0.50 + 0.50 * d_mean
                        edge_penalty = (1 - alpha) * pen_lo + alpha * pen_hi
                        sam_term *= edge_penalty
                    else:
                        edge_penalty = 1.0
                else:
                    edge_penalty = 1.0

                final = (clip_w * sims['final'] + sam_term + prior_w * prior)

                if qinfo['type'] == 'comparison':
                    final += area_w * area_norm

                if ct_circle is not None:
                    cx, cy, r = ct_circle
                    bbox = bbox_from_mask(m['segmentation'])
                    inside_ratio = rect_intersect_ratio_with_circle_bbox(bbox, h, w, (cx, cy), r)
                    final *= (0.7 + 0.3 * inside_ratio)

                scored.append({
                    'mask': m['segmentation'],
                    'bbox': bbox_from_mask(m['segmentation']),
                    'area': area,
                    'sam_score': sam_score,
                    'sam_score_original': m.get('sam_score_original', sam_score),
                    'clip_tight': sims.get('tight_pos', 0.0),
                    'clip_padded': sims.get('padded_pos', 0.0),
                    'clip_soft': sims.get('soft_pos', 0.0),
                    'clip_final': sims['final'],
                    'position_prior': prior,
                    'final_score': float(final),
                    'source': m['source'],
                    'edge_penalty': edge_penalty,
                    'weights_used': {'clip': clip_w, 'sam': sam_w, 'prior': prior_w, 'area': area_w},
                })

                if i < 3:
                    print(f"[Score] mask {i+1}: SAM={sam_score:.3f} | CLIP={sims['final']:.3f} | "
                          f"prior={prior:.3f} | edge_pen={edge_penalty:.3f} | final={final:.3f}")

            except Exception as e:
                print(f"[Score] failed for mask {i}: {e}")
                continue

        scored = sorted(scored, key=lambda x: x['final_score'], reverse=True)
        if len(scored) >= 2 and self.cfg.enable_uncertainty_detection:
            gap = scored[0]['final_score'] - scored[1]['final_score']
            print(f"[Score] top gap: {gap:.3f} -> {'confident' if gap >= self.cfg.uncertainty_threshold else 'uncertain'}")

        topk = scored[:self.cfg.top_k_display]
        result = {
            'sample_id': sample_id, 'image_path': image_path, 'question': question, 'answer': answer,
            'content_type': content_type, 'question_info': qinfo, 'prompts': prompts,
            'negative_prompts': negative_prompts, 'top_masks': topk, 'image_shape': (h, w),
            'grid_points': self.medsam.grid_points_viz, 'prompt_boxes': self.medsam.prompt_boxes_viz,
            'body_mask_info': self.medsam.body_mask_info
        }
        self._visualize(bgr, rgb_for_sam, body_mask, safe_mask, dist, result, ct_circle)
        self._print_summary(all_masks=scored, top_masks=topk, qinfo=qinfo)
        return result

    def _print_summary(self, all_masks: List[Dict], top_masks: List[Dict], qinfo: Dict):
        print("[Summary]")
        by_src = {}
        for m in all_masks:
            by_src[m['source']] = by_src.get(m['source'], 0) + 1
        if by_src:
            tot = len(all_masks)
            msg = ", ".join([f"{k}={v} ({v/tot*100:.1f}%)" for k, v in by_src.items()])
            print(f"[Summary] source distribution: {msg}")
        if top_masks:
            m0 = top_masks[0]
            print(f"[Summary] top-1: final={m0['final_score']:.3f} | SAM={m0['sam_score']:.3f} | "
                  f"CLIP={m0['clip_final']:.3f} | prior={m0['position_prior']:.3f} | source={m0['source']}")

    def _visualize(self, bgr: np.ndarray, rgb: np.ndarray, body_mask: np.ndarray, safe_mask: np.ndarray,
                   dist: np.ndarray, result: Dict, ct_circle: Optional[Tuple[int,int,int]]):
        h, w = bgr.shape[:2]; img_rgb = to_rgb(bgr)
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes[0, 0].imshow(img_rgb); axes[0, 0].set_title("Original + Grid + Boxes"); axes[0, 0].axis('off')
        if result['grid_points'] is not None and len(result['grid_points']) > 0:
            P = result['grid_points']; axes[0, 0].scatter(P[:, 0], P[:, 1], c='red', s=12, alpha=0.8, marker='x')
        if result.get('prompt_boxes'):
            for b in result['prompt_boxes']:
                x0, y0, x1, y1 = b
                axes[0, 0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=1.5,
                                                   edgecolor='yellow', facecolor='none', alpha=0.7))
        cnts, _ = cv2.findContours((body_mask.astype(np.uint8) * 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cnts:
            if c.shape[0] > 3:
                c = c.squeeze(); axes[0, 0].plot(c[:, 0], c[:, 1], color='lime', linewidth=1.5, alpha=0.7)
        cnts, _ = cv2.findContours((safe_mask.astype(np.uint8) * 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in cnts:
            if c.shape[0] > 3:
                c = c.squeeze(); axes[0, 0].plot(c[:, 0], c[:, 1], color='cyan', linewidth=1.2, alpha=0.7)
        if ct_circle is not None:
            cx, cy, r = ct_circle
            axes[0, 0].add_patch(plt.Circle((cx, cy), r, color='cyan', fill=False, linewidth=1.5, alpha=0.6))

        axes[0, 1].imshow(img_rgb); axes[0, 1].axis('off')
        if result['top_masks']:
            m0 = result['top_masks'][0]
            axes[0, 1].set_title(f"Top-1: {m0['final_score']:.3f} ({m0['source']})")
            overlay = img_rgb.copy()
            mk = m0['mask']; mask_col = np.zeros_like(img_rgb); mask_col[mk] = [0, 255, 0]
            overlay = cv2.addWeighted(overlay, 0.7, mask_col, 0.3, 0)
            axes[0, 1].imshow(overlay)
            x, y, w0, h0 = m0['bbox']
            axes[0, 1].add_patch(plt.Rectangle((x, y), w0, h0, linewidth=2, edgecolor='blue', facecolor='none'))

        axes[0, 2].imshow(img_rgb); axes[0, 2].axis('off'); axes[0, 2].set_title("Top-3 Regions")
        colors = ['red', 'green', 'blue']
        for i, m in enumerate(result['top_masks'][:3]):
            x, y, w0, h0 = m['bbox']
            axes[0, 2].add_patch(plt.Rectangle((x, y), w0, h0, linewidth=2, edgecolor=colors[i % 3], facecolor='none'))
            axes[0, 2].text(x, y - 5, f"#{i+1}:{m['final_score']:.3f}({m['source']})",
                            fontsize=9, color=colors[i % 3], weight='bold')

        if result['top_masks']:
            src_count = {}
            for m in result['top_masks'][:10]:
                src_count[m['source']] = src_count.get(m['source'], 0) + 1
            if len(src_count) > 1:
                axes[1, 0].pie(src_count.values(), labels=src_count.keys(), autopct='%1.1f%%')
                axes[1, 0].set_title("Source Distribution (Top-10)")
            else:
                axes[1, 0].text(0.5, 0.5, f"Single source:\n{list(src_count.keys())[0]}",
                                ha='center', va='center', transform=axes[1, 0].transAxes)
                axes[1, 0].set_title("Source Distribution")
        else:
            axes[1, 0].set_title("No Results")

        axes[1, 1].set_title("Score Components (Top-1)")
        if result['top_masks']:
            m0 = result['top_masks'][0]
            comps = {'CLIP\nFinal': m0.get('clip_final', 0), 'SAM\nScore': m0.get('sam_score', 0),
                     'Prior': m0.get('position_prior', 0)}
            xp = np.arange(len(comps)); bars = axes[1, 1].bar(xp, list(comps.values()))
            axes[1, 1].set_xticks(xp); axes[1, 1].set_xticklabels(list(comps.keys())); axes[1, 1].set_ylim([0, 1])
            for b, v in zip(bars, comps.values()):
                axes[1, 1].text(b.get_x() + b.get_width() / 2., v + 0.02, f'{v:.3f}',
                                ha='center', va='bottom', fontsize=9)

        axes[1, 2].set_title("CLIP Windows (Top-1)")
        if result['top_masks']:
            m0 = result['top_masks'][0]
            wins = {'Tight': m0.get('clip_tight', 0), 'Padded': m0.get('clip_padded', 0), 'Soft': m0.get('clip_soft', 0)}
            xp = np.arange(len(wins)); bars = axes[1, 2].bar(xp, list(wins.values()))
            axes[1, 2].set_xticks(xp); axes[1, 2].set_xticklabels(list(wins.keys())); axes[1, 2].set_ylim([0, 1])

        qtext = result['question'][:60] + "..." if len(result['question']) > 60 else result['question']
        qtype = result['question_info'].get('type', 'unknown')
        strategy = result['question_info'].get('strategy', 'unknown')
        fig.suptitle(f"Q: {qtext}\nA: {result['answer']} | Type: {qtype} | Strategy: {strategy}", fontsize=11, y=0.98)
        plt.tight_layout(); plt.show()

In [ ]:
# =============================== Test Helpers ===============================

def run_quick_batch(system, slake_path: str, start: int, end: int,
                    filter_content_type: Optional[str]=None):
    test_json = Path(slake_path) / "test.json"
    data = json.load(open(test_json, 'r'))
    eng = [s for s in data if s.get('q_lang') == 'en']
    if filter_content_type:
        samples = [s for s in eng if s.get('content_type') == filter_content_type]
    else:
        samples = [s for s in eng if s.get('content_type') != 'Modality']
    results = []
    for idx in range(start, min(end, len(samples))):
        s = samples[idx]
        img_path = str(Path(slake_path) / "imgs" / s['img_name'])
        try:
            r = system.process_single_sample(img_path, s['question'], s['answer'],
                                              sample_id=idx, content_type=s.get('content_type'))
            if r:
                results.append(r)
        except Exception:
            import traceback; traceback.print_exc()
    return results

MedicalVQAGrounding.test_range = run_quick_batch  # bind as method

In [ ]:
# =============================== Run ========================================
# Set these two paths before running.
#   SLAKE_PATH        : SLAKE dataset root, containing imgs/ and test.json
#   MEDSAM_CHECKPOINT : MedSAM vit_b checkpoint (.pth); see README for download

SLAKE_PATH = "/path/to/SLAKE/Slake1.0"
MEDSAM_CHECKPOINT = "/path/to/medsam_vit_b.pth"

cfg = GroundingConfig()
cfg.sam_score_threshold = 0.20
cfg.min_area_ratio = 0.001
cfg.use_three_windows = True
cfg.use_negative_evidence = True
cfg.use_source_calibration = True
cfg.enable_fallback = True

try:
    system = MedicalVQAGrounding(MEDSAM_CHECKPOINT, cfg)
    results = system.test_range(SLAKE_PATH, 0, 45)
    print(f"\nDone. Samples processed: {len(results)}")
except Exception as e:
    print(f"\nFailed: {e}")
    import traceback; traceback.print_exc()